# Song Recommender System

I'm building a content-based song recommender. The idea is to find songs that sound similar based on their audio features like danceability, energy, and tempo. This notebook will take us through the data pipeline, EDA, and finally the recommendation engine.

### Step 1: Loading our libraries

I'll start by importing the standard data science stack. I'm using `scikit-learn` for all the ML heavy lifting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# Let's make the plots look a bit nicer
sns.set_theme(style="whitegrid")

### Step 2: Getting the data

I'm loading the `dataset.csv` file. I noticed it already has an index column at the start, so I'll make sure to set `index_col=0` so I don't get an extra "Unnamed" column.

In [ ]:
df = pd.read_csv('dataset.csv', index_col=0)
df.head()

### Step 3: Cleaning things up

I checked and there is only one row with missing values, so I'll just drop it. Most importantly, I'm going to deduplicate the tracks. Since the same song can appear on multiple albums (like a single and a compilation), I'll keep the one with the highest popularity to make sure we're using the most representative version.

In [ ]:
# Dropping the single NaN row
df.dropna(inplace=True)

# Sorting by popularity to keep the most popular version during deduplication
df = df.sort_values('popularity', ascending=False)
df = df.drop_duplicates(subset=['track_name', 'artists'], keep='first')

# Simple cleanup for the genre strings
df['track_genre'] = df['track_genre'].str.lower().str.strip()

print(f"Data cleaned! We now have {len(df)} unique tracks.")
df.head()

### Step 4: Feature Engineering

This is the core of the data preparation. I need to convert everything into a numerical format that the recommender can understand. 

1. **Scaling Audio Features**: Numerical features like `loudness` and `tempo` are on completely different scales. I'll use `MinMaxScaler` to squeeze them all into a [0, 1] range so they contribute equally to the similarity score.
2. **Encoding Genres**: Since `track_genre` is categorical, I'll use `OneHotEncoder`. This creates a binary column for every genre, which is a great comprehensive way to handle categories in a distance-based model.

In [ ]:
# 1. Scaling the numerical audio features
audio_features = ['danceability', 'energy', 'key', 'loudness', 'mode', 
                  'speechiness', 'acousticness', 'instrumentalness', 
                  'liveness', 'valence', 'tempo']

scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df[audio_features]), 
                         columns=audio_features, 
                         index=df.index)

# 2. One-hot encoding the genres
encoder = OneHotEncoder(sparse_output=False)
genre_encoded = encoder.fit_transform(df[['track_genre']])
genre_columns = encoder.get_feature_names_out(['track_genre'])
df_genres = pd.DataFrame(genre_encoded, columns=genre_columns, index=df.index)

# 3. Bringing it all together
df_final = pd.concat([df_scaled, df_genres], axis=1)

print(f"Feature matrix created with shape: {df_final.shape}")
df_final.head()